In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="Versions des packages">

Le code de cette page a été développé avec les exigences suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Tu peux utiliser les options pour personnaliser la primitive Estimator. Bien que l'interface de la méthode `run()` des primitives soit commune à toutes les implémentations, leurs options ne le sont pas. Consulte les références API pour obtenir des informations sur les options de [`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) et [`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html).

<span id="pass-options"></span>

## Définir les options de l'Estimator

Tu peux définir les options lors de l'initialisation de l'Estimator, après l'initialisation de l'Estimator, ou tu peux mettre à jour les options après que l'Estimator a été initialisé. Pour les instructions sur l'utilisation de ces techniques, consulte le sujet [Introduction aux options](/guides/runtime-options-overview#options-precedence).

De plus, tu peux définir la valeur `precision` dans la méthode `run()`, comme décrit dans la section suivante.
<span id="run-method"></span>

### Méthode Run()

Les seules valeurs que tu peux passer à `run()` sont celles définies dans l'interface. C'est-à-dire `precision` pour Estimator. Cela écrase toute valeur définie pour `default_precision` pour l'exécution courante.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Cas particulier : précision
La méthode `EstimatorV2.run` accepte deux arguments : une liste de PUBs, chacun pouvant spécifier une valeur de précision spécifique au PUB, et un argument de mot-clé precision. Ces valeurs de précision font partie de l'interface d'exécution de l'Estimator, et sont indépendantes des options du Runtime Estimator. Elles ont la priorité sur toutes les valeurs spécifiées en tant qu'options afin de se conformer à l'abstraction de l'Estimator.

Cependant, si `precision` n'est pas spécifié par un PUB ou dans l'argument de mot-clé run (ou s'ils sont tous `None`), alors la valeur de précision des options est utilisée, notamment `default_precision`.

Note que les options de l'Estimator contiennent à la fois `default_shots` et `default_precision`. Cependant, comme le gate-twirling est activé par défaut, le produit de `num_randomizations` et `shots_per_randomization` prend la priorité sur ces deux options.

Spécifiquement, pour tout PUB d'Estimator :

1. Si le PUB spécifie la précision, utiliser cette valeur.
2. Si l'argument de mot-clé precision est spécifié dans `run`, utiliser cette valeur.
3. Si `twirling` est activé (True par défaut), alors le produit de `num_randomizations` et `shots_per_randomization`, tel que spécifié dans les [options `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options), est utilisé.
4. Si `estimator.options.default_shots` est spécifié, utiliser cette valeur pour contrôler la quantité de données.
5. Si `estimator.options.default_precision` est spécifié, utiliser cette valeur.

Par exemple, si la précision est spécifiée dans les quatre endroits, celui avec la priorité la plus haute (précision spécifiée dans le PUB) est utilisé.

> **Note:** La précision évolue inversement avec l'utilisation. C'est-à-dire que plus la précision est faible, plus le temps QPU nécessaire à l'exécution est long.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## Désactiver toute la mitigation d'erreurs et la suppression d'erreurs
Tu peux désactiver toute la mitigation et la suppression d'erreurs si tu effectues, par exemple, des recherches sur tes propres techniques de mitigation. Pour ce faire, définis `resilience_level = 0`.

Exemple :

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## Options disponibles
Le tableau suivant documente les options de la dernière version de `qiskit-ibm-runtime`. Pour voir les versions d'options antérieures, consulte la [référence API `qiskit-ibm-runtime`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime) et sélectionne une version précédente.

<Accordion>
<AccordionItem title="`default_shots`">

Le nombre total de shots à utiliser par circuit et par configuration.

**Choix** : Entier >= 0

**Par défaut** : None

[Documentation API `default_shots`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

La précision par défaut à utiliser pour tout PUB ou appel `run()` qui n'en spécifie pas une.

**Choix** : Float > 0

**Par défaut** : 0.015625 (1 / sqrt(4096))

[Documentation API `default_precision`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

Contrôler les paramètres de la mitigation d'erreur par découplage dynamique.

[Documentation API `dynamical_decoupling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choix** : `True`, `False`

**Par défaut** : `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choix** : `middle`, `edges`

**Par défaut** : `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choix : `asap`, `alap`
Par défaut : `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choix : `XX`, `XpXm`, `XY4`
Par défaut : `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choix : `True`, `False`
Par défaut : `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[Documentation API `environment`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Fonction callable qui reçoit le `Job ID` et le `Job result`.

**Choix** : None

**Par défaut** : None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

Liste de balises.

**Choix** : None

**Par défaut** : None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**Choix** : DEBUG, INFO, WARNING, ERROR, CRITICAL

**Par défaut** : WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**Choix** : `True`, `False`

**Par défaut** : `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[Documentation API `execution`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Indique s'il faut réinitialiser les qubits à l'état fondamental pour chaque shot.

**Choix** : `True`, `False`

**Par défaut** : `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
Le délai entre une mesure et le circuit quantique suivant.

**Choix** : Valeur dans la plage fournie par `backend.rep_delay_range`

**Par défaut** : Donné par `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
Limite la durée d'exécution d'un job, en secondes. Consulte le guide sur le [temps d'exécution maximum](/guides/max-execution-time) pour plus de détails.

**Choix** : Nombre entier de secondes dans la plage [1, 10800]

**Par défaut** : 10800 (3 heures)
  </AccordionItem>

<AccordionItem title="`resilience`">
Options de résilience avancées pour affiner la stratégie de résilience.

[Documentation API `resilience`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options pour l'apprentissage du bruit des couches.

[Documentation API `resilience.layer_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choix** : list[int] de 2 à 10 valeurs dans la plage [0, 200]

**Par défaut** : `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choix** : None, Entier >= 1

**Par défaut** : `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choix** : Entier >= 1

**Par défaut** : `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choix** : Entier >= 1

**Par défaut** : `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**Choix** : `NoiseLearnerResult`, `Sequence[LayerError]`

**Par défaut** : None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**Choix** : `True`, `False`

**Par défaut** : `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

Options pour l'apprentissage du bruit de mesure.

[Documentation API `resilience.measure_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choix** : Entier >= 1

**Par défaut** : `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choix** : Entier, `auto`

**Par défaut** : `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**Choix** : `True`, `False`

**Par défaut** : `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

Options de mitigation par cancellation d'erreurs probabiliste.

[Documentation API `resilience.pec`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**Choix** : `None`, Entier >= 1

**Par défaut** : `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**Choix** : `auto`, float dans la plage [0, 1]

**Par défaut** : `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**Choix** : `True`, `False`

**Par défaut** : `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[Documentation API `resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**Choix** : `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Par défaut** : `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choix** : Liste de floats

**Par défaut** : `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**Choix** : Un ou plusieurs de : `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Par défaut** : `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**Choix** : Liste de floats ; chaque float >= 1

**Par défaut** : `(1, 1.5, 2)` pour `PEA`, et `(1, 3, 5)` sinon

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

Niveau de résilience à construire contre les erreurs. Des niveaux plus élevés génèrent des résultats plus précis au détriment de temps de traitement plus longs. Consulte la section [niveaux de résilience](/guides/estimator-noise-management#resilience) dans le sujet Gestion du bruit pour en savoir plus.

**Choix** : `0`, `1`, `2`

**Par défaut** : `1`

[Documentation API `resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**Choix** : Entier

**Par défaut** : None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

Options à passer lors de la simulation d'un Backend

[Documentation API `simulator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choix** : Liste des noms de portes de base à déplier

**Par défaut** : L'ensemble de toutes les portes de base supportées par le [simulateur Qiskit Aer](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**Choix** : Liste d'interactions à deux qubits dirigées

**Par défaut** : None, ce qui implique l'absence de contraintes de connectivité (connectivité complète).

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**Choix** : [Qiskit Aer NoiseModel](/guides/build-noise-models), ou sa représentation

**Par défaut** : None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**Choix** : Entier

**Par défaut** : None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

Options de twirling

[Documentation API `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choix** : True, False

**Par défaut** : False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**Choix** : True, False

**Par défaut** : True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**Choix** : `auto`, Entier >= 1

**Par défaut** : `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**Choix** : `auto`, Entier >= 1

**Par défaut** : `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**Choix** : `active`, `active-circuit`, `active-accum`, `all`

**Par défaut** : `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

Options expérimentales, le cas échéant.

  </AccordionItem>

</Accordion>
<span id="options-compatibility-table"></span>
## Compatibilité des fonctionnalités
Certaines fonctionnalités de runtime ne peuvent pas être utilisées ensemble dans un seul job. Clique sur l'onglet approprié pour obtenir la liste des fonctionnalités incompatibles avec la fonctionnalité sélectionnée :

<Tabs>

  <TabItem value="Fractional" label="Fractional gates">
  Incompatible avec :
  - Gate twirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="Gate-folding ZNE">
    Incompatible avec :
  - PEA
  - PEC

  Peut ne pas fonctionner lors de l'utilisation de gates personnalisés.
  </TabItem>
  <TabItem value="Twirling" label="Gate twirling">
  Incompatible avec les fractional gates ou les stretches.

  Autres remarques :
  - Le measurement twirling ne peut être appliqué qu'aux mesures terminales.
  - Ne fonctionne pas avec les entangleurs non-Clifford.

  </TabItem>

  <TabItem value="PEA" label="PEA">
    Incompatible avec :
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    Incompatible avec :
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </TabItem>

</Tabs>
## Prochaines étapes
> **Tip:** - Consulte le guide [Introduction aux options](/guides/runtime-options-overview).
> - Trouve plus de détails sur les méthodes `EstimatorV2` dans la [référence API Estimator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2).
> - Décide dans quel [mode d'exécution](/guides/execution-modes) exécuter ton job.
> - Découvre la [gestion du bruit avec Estimator](/guides/estimator-noise-management).

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>